# Code analysis and git graphs

What a cloned repository is made of, and how its history moved across branches - on this
repository itself, so the notebook needs no network. Languages, grammars and framework rules
come from `resources/` (see the docs guide *Analysing code*).

In [ ]:
from gitrecon.code import analyze_repo
from gitrecon.config import PROJECT_ROOT

result = analyze_repo(PROJECT_ROOT)
result["main_language"], result["files"], result["frameworks"][:8]

In [ ]:
for name, entry in list(result["languages"].items())[:6]:
    print(f"{name:<18} {entry['files']:>4} files  code {entry.get('code', 0):>6}  comments {entry.get('comment', 0):>5}")
result["python"]["functions"], result["python"]["classes"], list(result["python"]["imports"])[:8]

## The lexer by itself

One Lark grammar per language family tells comments, strings, numbers and names apart; it
never fails on syntax it does not know.

In [ ]:
from pathlib import Path

from gitrecon.code import lexical
from gitrecon.code.languages import detect

source = 'const url = "https://example.com // not a comment";  // a comment\nfunction fetchData(count) { return count; }'
stats = lexical.lex(source, detect(Path("demo.js")))
dict(stats.tokens), dict(stats.names), stats.urls, (stats.code, stats.comment)

## History as a Mermaid gitGraph

Every branch claims the commits on its first-parent chain (a *lane*); merges become `merge`.

In [ ]:
from collections import Counter

from gitrecon.mapping.gitgraph import assign_lanes, branch_tips, git_graph, read_history

commits = read_history(PROJECT_ROOT, max_commits=40)
assign_lanes(commits, branch_tips(PROJECT_ROOT))
print(Counter(c.lane for c in commits))
print(git_graph(PROJECT_ROOT, max_commits=12))

CLI: `gitrecon analyze [PATH...]`, `gitrecon gitgraph [PATH] --format mermaid --save`
(saved to `data/gitgraphs/`).